# Fundamentos de PyTorch

## El ecosistema moderno de Deep Learning

Para entender el panorama actual del ML, es útil ver los principales frameworks como **herramientas especializadas** para distintas etapas del desarrollo.

### Los dos pilares: PyTorch y TensorFlow

| | **PyTorch** (Meta) | **TensorFlow** (Google) |
|---|---|---|
| **Fortaleza** | Investigación e IA generativa | Producción empresarial |
| **Diseño** | Grafo dinámico, Pythónico | Grafo estático, servicio escalable |
| **Herramientas clave** | `torch.compile` (2.x), Hugging Face | TF Serving, TF Lite |
| **Base de usuarios** | Academia, investigadores de LLM/difusión | Pipelines industriales a gran escala |

**PyTorch** es el framework dominante en investigación e IA generativa. Su diseño intuitivo y Pythónico, con grafo de cómputo dinámico, lo convierte en la primera opción para prototipar nuevas arquitecturas. Con PyTorch 2.x y `torch.compile`, también ha cerrado la brecha de rendimiento para producción.

**TensorFlow** sigue siendo una potencia empresarial — TF Serving y TF Lite lo mantienen arraigado en pipelines de producción donde la estabilidad y la escalabilidad son prioritarias.

### Roles especializados

**JAX** (Google) es un framework de cómputo numérico de alto rendimiento usado por laboratorios de investigación de élite (DeepMind, Anthropic) para entrenar modelos masivos en clusters de TPU. Su estilo de programación funcional lo mantiene fuera de los cursos básicos para principiantes.

**Keras 3** es el *gran unificador*: una API de alto nivel que corre sobre PyTorch, TensorFlow o JAX. Escribe una vez, cambia de backend cuando quieras.

<img src="../img/keras_on_top.png" width="300" alt="Keras corre sobre múltiples backends">

### Adopción de frameworks a lo largo del tiempo

<img src="../img/framework_trends_2026.png" width="600" alt="Tendencias de frameworks">

## Instalación

Sigue las [instrucciones oficiales](https://pytorch.org/get-started/locally/) para instalar PyTorch en tu plataforma. El instalador te permite elegir el backend de cómputo:

- **GPUs NVIDIA**: instala con soporte CUDA
- **GPUs AMD (Linux)**: usa [ROCm](https://pytorch.org/blog/pytorch-for-amd-rocm-platform-now-available-as-python-package/)
- **Apple Silicon**: MPS (Metal Performance Shaders) está integrado
- **Solo CPU**: funciona en cualquier lugar, solo más lento

Con `uv`: `uv add torch torchvision`

In [9]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.11.0
CUDA available: False


## Tensores

Los tensores son la estructura de datos fundamental en PyTorch — arreglos multidimensionales, como los arreglos de NumPy, pero con dos superpoderes:

1. **Aceleración por GPU**: los tensores pueden vivir en CPU o GPU, permitiendo cómputo paralelo
2. **Diferenciación automática**: PyTorch rastrea las operaciones sobre tensores para calcular gradientes

Los tensores pueden representar muchos tipos de datos:
- Escalar (0D): un solo número
- Vector (1D): una lista de características
- Matriz (2D): un batch de vectores de características, o una imagen en escala de grises
- Tensor 3D: una imagen RGB (canales × alto × ancho)
- Tensor 4D: un batch de imágenes (batch × canales × alto × ancho)

[![tensores](../img/tensors.png)](https://medium.com/@anoorasfatima/10-most-common-maths-operation-with-pytorchs-tensor-70a491d8cafd)

### Creación de tensores

In [10]:
import torch
import numpy as np

# Desde una lista de Python
t1 = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(f"Tensor:\n{t1}")
print(f"Tipo: {t1.dtype}")   # int64 (inferido)
print(f"Forma: {t1.shape}")  # torch.Size([2, 3])

Tensor:
tensor([[1, 2, 3],
        [4, 5, 6]])
Tipo: torch.int64
Forma: torch.Size([2, 3])


In [11]:
# Desde NumPy (comparte memoria — cambios en uno afectan al otro)
arr = np.array([1.0, 2.0, 3.0])
t_from_np = torch.from_numpy(arr)
print(f"Desde NumPy: {t_from_np}")

Desde NumPy: tensor([1., 2., 3.], dtype=torch.float64)


In [12]:
# Funciones de creación comunes (misma API que NumPy)
print("Ceros:", torch.zeros(2, 3))
print("Unos:", torch.ones(2, 3))
print("Aleatorio:", torch.rand(2, 3))
print("Rango:", torch.arange(0, 10, 2))

Ceros: tensor([[0., 0., 0.],
        [0., 0., 0.]])
Unos: tensor([[1., 1., 1.],
        [1., 1., 1.]])
Aleatorio: tensor([[0.0478, 0.6082, 0.4808],
        [0.7491, 0.2647, 0.8708]])
Rango: tensor([0, 2, 4, 6, 8])


In [13]:
# dtype explícito
t_float = torch.tensor([1, 2, 3], dtype=torch.float32)
print(f"Tensor float32: {t_float}")

Tensor float32: tensor([1., 2., 3.])


### Operaciones

In [14]:
a = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)
b = torch.tensor([[5, 6], [7, 8]], dtype=torch.float32)

# Operaciones elemento a elemento
print("a + b =", a + b)
print("a * b =", a * b)

# Multiplicación de matrices (dos formas equivalentes)
print("a @ b =", a @ b)
print("matmul:", torch.matmul(a, b))

# Transpuesta
print("a.T =", a.T)

a + b = tensor([[ 6.,  8.],
        [10., 12.]])
a * b = tensor([[ 5., 12.],
        [21., 32.]])
a @ b = tensor([[19., 22.],
        [43., 50.]])
matmul: tensor([[19., 22.],
        [43., 50.]])
a.T = tensor([[1., 3.],
        [2., 4.]])


### Indexación

In [15]:
t = torch.tensor([[10, 20, 30], [40, 50, 60], [70, 80, 90]])

print("Primera fila:", t[0])
print("Elemento [1,2]:", t[1, 2])
print("Segunda columna:", t[:, 1])
print("Submatriz:", t[1:, 1:])

Primera fila: tensor([10, 20, 30])
Elemento [1,2]: tensor(60)
Segunda columna: tensor([20, 50, 80])
Submatriz: tensor([[50, 60],
        [80, 90]])


## Uso de GPU

Las GPUs aceleran dramáticamente el entrenamiento de redes neuronales gracias al paralelismo masivo (**GPGPU** — Cómputo de propósito general en unidades de procesamiento gráfico).

PyTorch soporta nativamente:
- **GPUs NVIDIA** vía [CUDA](https://en.wikipedia.org/wiki/CUDA)
- **Apple Silicon** vía [Metal (MPS)](https://en.wikipedia.org/wiki/Metal_(API))
- **GPUs AMD** vía [ROCm](https://pytorch.org/blog/pytorch-for-amd-rocm-platform-now-available-as-python-package/)

Los tensores deben estar en el **mismo dispositivo** antes de poder combinarlos:

In [16]:
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Usando dispositivo: {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Usando dispositivo: mps


In [17]:
# Mover tensores entre dispositivos
t_cpu = torch.rand(3, 3)
print(f"Dispositivo: {t_cpu.device}")

t_device = t_cpu.to(device)
print(f"Después de .to('{device}'): {t_device.device}")

# Las operaciones requieren el mismo dispositivo — esto fallaría:
# torch.matmul(t_cpu, t_device)  # RuntimeError!

# Mover ambos al mismo dispositivo primero
result = torch.matmul(t_cpu.to(device), t_device)
print(f"Resultado en {result.device}: {result}")

Dispositivo: cpu
Después de .to('mps'): mps:0
Resultado en mps:0: tensor([[0.5435, 0.4876, 0.2255],
        [0.9860, 0.7554, 0.6070],
        [1.3470, 0.9821, 0.7058]], device='mps:0')


## Construir modelos con `nn.Module`

`torch.nn.Module` es la clase base de toda red neuronal en PyTorch. Para crear un modelo:

1. **Hereda** de `nn.Module`
2. Define las capas (Layers) en `__init__`
3. Define cómo fluyen los datos en `forward`

PyTorch rastrea automáticamente todos los objetos `nn.Parameter` (Weights, Biases) registrados en `__init__`, por lo que puedes iterarlos con `.parameters()`:

In [18]:
import torch.nn as nn

class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(100, 200)
        self.activation = nn.ReLU()
        self.linear2 = nn.Linear(200, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)
        return x

model = TinyModel()
print(model)
print(f"\nTotal de parámetros: {sum(p.numel() for p in model.parameters()):,}")

TinyModel(
  (linear1): Linear(in_features=100, out_features=200, bias=True)
  (activation): ReLU()
  (linear2): Linear(in_features=200, out_features=10, bias=True)
)

Total de parámetros: 22,210


In [19]:
# Puedes inspeccionar capas individuales y sus parámetros
print("Solo la segunda capa:")
print(model.linear2)
print(f"  Forma del Weight: {model.linear2.weight.shape}")
print(f"  Forma del Bias:   {model.linear2.bias.shape}")

Solo la segunda capa:
Linear(in_features=200, out_features=10, bias=True)
  Forma del Weight: torch.Size([10, 200])
  Forma del Bias:   torch.Size([10])


## Estilo funcional: `torch.nn.functional`

El módulo `torch.nn.functional` ofrece las mismas operaciones que las capas basadas en módulos, pero como **funciones sin estado**. Esto es útil para operaciones sin parámetros aprendibles (como las Activations):

In [20]:
import torch.nn.functional as F

class TinyModelFunctional(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(100, 200)
        self.linear2 = nn.Linear(200, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.linear1(x))  # Activation funcional
        x = self.linear2(x)
        return x

# Ambos estilos producen el mismo resultado — es una cuestión de preferencia

En este caso, la biblioteca [`nn.functional`](https://pytorch.org/tutorials/beginner/nn_tutorial.html#using-torch-nn-functional) se usa para llamar a las Activation Functions directamente. Es otro estilo de programación. En ambos casos se llaman desde `forward`, pero la diferencia es que en este caso no es necesario crear un atributo de clase `nn.ReLU()` en el constructor.

## `nn.Sequential`

Para modelos simples donde las capas (Layers) se aplican una tras otra, `nn.Sequential` ahorra código repetitivo. Ni siquiera necesitas un método `forward`:

In [21]:
# Estas tres definiciones son equivalentes:

# 1. Sequential (más simple)
model_seq = nn.Sequential(
    nn.Linear(100, 200),
    nn.ReLU(),
    nn.Linear(200, 10),
)

# 2. Sequential con capas nombradas (útil para depuración)
from collections import OrderedDict

model_named = nn.Sequential(OrderedDict([
    ("fc1", nn.Linear(100, 200)),
    ("relu", nn.ReLU()),
    ("fc2", nn.Linear(200, 10)),
]))

# Ambas funcionan igual:
x = torch.rand(1, 100)
print(f"Forma de salida Sequential: {model_seq(x).shape}")
print(f"Forma de salida nombrado:   {model_named(x).shape}")

Forma de salida Sequential: torch.Size([1, 10])
Forma de salida nombrado:   torch.Size([1, 10])


## Capas lineales en profundidad

Una capa lineal (totalmente conectada) calcula:

$$y = xW^T + b$$

donde $W$ es la matriz de Weights y $b$ es el vector de Bias. Si la capa tiene $m$ entradas y $n$ salidas, entonces $W$ tiene forma $(n, m)$ y $b$ tiene forma $(n,)$.

Por ejemplo, con 3 entradas y 2 salidas:

$$\begin{bmatrix} y_1 \\ y_2 \end{bmatrix} = \begin{bmatrix} w_{11} & w_{12} & w_{13} \\ w_{21} & w_{22} & w_{23} \end{bmatrix} \begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix} + \begin{bmatrix} b_1 \\ b_2 \end{bmatrix}$$

In [22]:
# Verificar la matemática manualmente
lin = nn.Linear(3, 2)

x = torch.tensor([[1.0, 2.0, 3.0]])
y = lin(x)

# Cómputo manual: y = x @ W^T + b
y_manual = x @ lin.weight.T + lin.bias
print(f"Salida de la capa:  {y.detach()}")
print(f"Salida manual: {y_manual.detach()}")
print(f"Coinciden: {torch.allclose(y, y_manual)}")

Salida de la capa:  tensor([[-2.1286,  1.4348]])
Salida manual: tensor([[-2.1286,  1.4348]])
Coinciden: True


## Dataset y DataLoader

PyTorch ofrece dos abstracciones clave para el manejo de datos:

- **`torch.utils.data.Dataset`**: representa una colección de muestras. Debe implementar `__len__` y `__getitem__`.
- **`torch.utils.data.DataLoader`**: envuelve un Dataset y proporciona agrupación en Batches, mezcla aleatoria y carga en paralelo.

Así es como alimentas datos a tu ciclo de entrenamiento:

In [23]:
from torch.utils.data import Dataset, DataLoader

# Un Dataset personalizado mínimo
class SyntheticDataset(Dataset):
    def __init__(self, n_samples: int = 1000, n_features: int = 10):
        self.X = torch.randn(n_samples, n_features)
        self.y = (self.X.sum(dim=1) > 0).long()  # Etiqueta binaria

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.X[idx], self.y[idx]

dataset = SyntheticDataset(n_samples=200)
print(f"Tamaño del Dataset: {len(dataset)}")
print(f"Una muestra: X shape={dataset[0][0].shape}, y={dataset[0][1]}")

Tamaño del Dataset: 200
Una muestra: X shape=torch.Size([10]), y=1


In [24]:
# DataLoader: gestiona el agrupado en Batches y la mezcla aleatoria
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Iterar sobre un Batch
batch_X, batch_y = next(iter(loader))
print(f"Forma del Batch X: {batch_X.shape}")  # (32, 10)
print(f"Forma del Batch y: {batch_y.shape}")  # (32,)
print(f"Etiquetas en el Batch: {batch_y}")

Forma del Batch X: torch.Size([32, 10])
Forma del Batch y: torch.Size([32])
Etiquetas en el Batch: tensor([1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1,
        0, 0, 1, 0, 0, 1, 0, 0])


**Parámetros clave:**
- `batch_size`: cuántas muestras por Batch (32–256 es típico)
- `shuffle=True`: aleatorizar el orden en cada Epoch (importante para entrenamiento, no para evaluación)
- `num_workers`: procesos de carga de datos en paralelo (usar 0 en Windows/macOS, 2–4 en Linux)

PyTorch también provee Datasets listos para usar (MNIST, FashionMNIST, CIFAR-10, …) vía `torchvision.datasets`. Usaremos uno en el siguiente notebook.

## Fuentes

- [PyTorch Tensors documentation](https://pytorch.org/docs/stable/tensors.html)
- [PyTorch nn.Module documentation](https://pytorch.org/docs/stable/generated/torch.nn.Module.html)
- [PyTorch Data utilities](https://pytorch.org/docs/stable/data.html)